In [1]:
!pip install ollama rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 30.9 MB/s eta 0:00:00


In [2]:
from ollama import Client
import sys
import os
import platform
from rdkit import Chem
from rdkit.Chem import QED, Draw

from google.colab import userdata
ollama_key = userdata.get('ollama_key')

In [3]:
scoring_args = [2.5, 'alogp']

def lipinski(smiles: str):
  '''
    calculates the aLogP of the molecule based on the scoring_args.
    If the calculation fails, returns -999.

    Args:
    - smiles (str): The SMILES string of the molecule to be scored.
    Returns:
    - score (float): The calculated aLogP score of the molecule, or -999 if the calculation fails.
  '''

  mol = Chem.MolFromSmiles(smiles)
  qed = Chem.QED.default(mol)

  p = Chem.QED.properties(mol)

  lipinski_hash = {'mw': 0, 'alogp': 1, 'hba': 2, 'hbd': 3, 'psa': 4, 'rb': 5, 'ar': 6, 'um': 7}

  try:
    score = p[lipinski_hash[scoring_args[1]]]
  except:
    score = -999

  return score

key = ollama_key

models = ['deepseek-v3.1:671b', 'gpt-oss:120b', 'gpt-oss:20b',
          'devstral-2:123b', 'cogito-2.1:671b',
          'nemotron-3-nano:30b', 'gemini-3-flash-preview',
          'kimi-k2:1t', 'kimi-k2.5']

model = models[-1]
client = Client(host = 'https://ollama.com',
            headers={'Authorization': f'Bearer {key}'})

available_functions = {
  'lipinski': lipinski
}

In [4]:
initial_input = input("Enter your question: ")
messages = [{'role': 'user', 'content': initial_input}]

while True:
    response = client.chat(
        model=model,
        messages=messages,
        tools=[lipinski],
        think=True,
    )
    messages.append(response.message)
    print('------------------------------------------------------------------------')
    print("Thinking: ", response.message.thinking)
    print('------------------------------------------------------------------------')
    print("Content: ", response.message.content)
    print('------------------------------------------------------------------------')
    if response.message.tool_calls:
        for tc in response.message.tool_calls:
            if tc.function.name in available_functions:
                print(f"Calling {tc.function.name} with arguments {tc.function.arguments}")
                result = available_functions[tc.function.name](**tc.function.arguments)
                print(f"Result: {result}")
                print('------------------------------------------------------------------------')
                # add the tool result to the messages
                messages.append({'role': 'tool', 'tool_name': tc.function.name, 'content': str(result)})
    else:
        # end the loop when there are no more tool calls
        break
  # continue the loop with the updated messages

Enter your question: Find a molecule based on c1ccc(O)c(O)c1 that has an alogP as close to 2.5 as possible
------------------------------------------------------------------------
Thinking:  The user wants me to find a molecule similar to c1ccc(O)c(O)c1 (which is catechol - a benzene ring with two hydroxyl groups on adjacent carbons) that has an alogP as close to 2.5 as possible.

First, let me calculate the alogP of the starting molecule c1ccc(O)c(O)c1 to see what we're working with.
------------------------------------------------------------------------
Content:  I'll help you find a molecule based on catechol (c1ccc(O)c(O)c1) with an alogP close to 2.5. Let me first check the alogP of the starting molecule and then explore some derivatives.
------------------------------------------------------------------------
Calling lipinski with arguments {'smiles': 'c1ccc(O)c(O)c1'}
Result: 1.0977999999999997
------------------------------------------------------------------------
-----------